# DD-PRiSM-plus — Step 1: set up and fetch all data

Run once in a **CPU** session. Click **Save Version** at the end or everything is lost.

**Session options (right-hand `<` panel):** Accelerator → **None**, Internet → **On**

> **figshare is down site-wide** (returns `202 Accepted` with an empty body,
> including for `figshare.com` itself). The two DepMap files come from there.
> Step 4 below copies them in from an attached Kaggle Dataset if you have
> supplied them manually; otherwise the script tries figshare and gives up fast.

In [ ]:
# 0. Session check
import subprocess, os, glob, shutil
ok = subprocess.run(['curl','-sI','--max-time','15','https://github.com'],
                    capture_output=True).returncode == 0
print('internet:', 'ON' if ok else 'OFF  <-- enable it, then rerun')
print('free disk:', subprocess.run(['df','-h','/kaggle/working'],
      capture_output=True, text=True).stdout.splitlines()[-1])

## 1. Get the code

In [ ]:
REPO = '/kaggle/working/ddprism-plus'
DATA = '/kaggle/working/data'

if os.path.exists(REPO):
    !cd {REPO} && git pull --quiet
else:
    !git clone --quiet https://github.com/SanaNiroomand/DD-PRiSM-plus.git {REPO}

os.chdir(REPO)
print('working in', os.getcwd())

## 2. Install what Kaggle lacks

`zipfile-deflate64` is **mandatory** — DOSERESP.zip is Deflate64 and the
standard library cannot decompress it.

In [ ]:
!pip install --quiet zipfile-deflate64 rdkit openpyxl
print('installed')

## 3. Check the model code (23 tests, ~4 s)

In [ ]:
!python -m pytest tests -q

## 4. Copy any manually-supplied files

If you downloaded the DepMap files in your browser and uploaded them as a
Kaggle Dataset, attach it with **Add Input → Datasets**, and this cell finds
them wherever they landed. Harmless if you have not.

In [ ]:
os.makedirs(DATA, exist_ok=True)

# accepted filenames -> the name the pipeline expects
ALIASES = {
    'OmicsExpressionProteinCodingGenesTPMLogp1.csv': 'OmicsExpressionProteinCodingGenesTPMLogp1.csv',
    'sample_info.csv':      'sample_info_18q3.csv',
    'sample_info_18q3.csv': 'sample_info_18q3.csv',
}

for found, wanted in ALIASES.items():
    target = os.path.join(DATA, wanted)
    if os.path.exists(target):
        continue
    hits = glob.glob(f'/kaggle/input/**/{found}', recursive=True)
    if hits:
        shutil.copy(hits[0], target)
        print(f'copied {wanted}  <-  {hits[0]}')

print('
contents of', DATA)
!ls -la {DATA} 2>/dev/null || echo '  (empty)'

## 5. Download everything else (~1 GB)

Files land **directly in `DATA`**. Already-present files are skipped, so this
is safe to rerun.

In [ ]:
!python scripts/get_data.py --dest {DATA} --include-optional --attempts 3

## 6. Retry stragglers (only if step 5 reported failures)

Worth a try each session in case figshare has recovered.

In [ ]:
!python scripts/get_data.py --dest {DATA} --only depmap_expression depmap_samples --attempts 4

## 7. Verify

Every required row must read `ok`. Size, archive integrity and — for the two
DepMap files — the official MD5 are all checked. This has caught four
different silent failures already, so do not skip it.

In [ ]:
!python scripts/get_data.py --dest {DATA} --check
print()
!du -sh {DATA}

## 8. Save it

**Save Version → Save & Run All (Commit).** Without this, everything here is
deleted when the session ends.

The next notebook attaches this via **Add Input → Your Work → Notebook Output**.

---

**Next:** preprocessing. Success is exactly **7,915,900** NCI60 training rows
and **1,387,317** combination rows.